In [ ]:
import json
import shutil
import pandas as pd

from pathlib import Path

In [ ]:
MILK10K_DOWNLOAD_PATH = Path("/home/sulcm/datasets/milk10k/milk10k_downloaded")
MILK10K_BUILDER = Path("/home/sulcm/datasets/milk10k/milk10k_builder")
MILK10K_DATASET = Path("/home/sulcm/datasets/milk10k/milk10k")

# Create dataset structure

In [ ]:
ds_gt = pd.read_csv(MILK10K_DOWNLOAD_PATH / "training_gt.csv").set_index("lesion_id", append=True)
labels = [l.lower() for l in ds_gt.columns.to_list()]
ds_gt_classes = ds_gt.dot(ds_gt.columns).apply(lambda x: x.lower())

In [ ]:
ds_training_input = pd.read_csv(MILK10K_DOWNLOAD_PATH / "milk10k" / "metadata.csv")
ds_training_input[["file_name", "label"]] = ds_training_input.apply(
    lambda row: [
        row["isic_id"] + ".jpg",
        ds_gt_classes.xs(row["lesion_id"], level=1).iloc[0]
    ],
    axis=1, result_type="expand"
)
ds_training_input.drop(columns=["attribution", "copyright_license"], inplace=True)

In [ ]:
ds_info = {
    "labels": labels
}

In [ ]:
if not MILK10K_BUILDER.exists():
    shutil.copytree(MILK10K_DOWNLOAD_PATH / "milk10k" / "images", MILK10K_BUILDER / "train")
    ds_training_input.to_csv(MILK10K_BUILDER / "train" / "metadata.csv", index=False)
    with open(MILK10K_BUILDER / "dataset_info.json", "w") as f:
        json.dump(ds_info, f, indent=2, ensure_ascii=False)

# Load/Build datatset

In [ ]:
from datasets import load_dataset, load_from_disk, ClassLabel

In [ ]:
dataset = load_dataset("imagefolder", data_dir=MILK10K_BUILDER)
dataset

In [ ]:
dataset = dataset.cast_column("label", ClassLabel(names=labels))

In [ ]:
dataset.save_to_disk(MILK10K_DATASET)

In [ ]:
lds = load_from_disk(
    dataset_path=MILK10K_DATASET
)

In [ ]:
lds["train"].features